# 04b — Tracker Architecture Assessment: ByteTrack

Companion to `04_tracker_architecture_botsort.ipynb`.
Runs the **same diagnostic and sweep pipeline** for **ByteTrack** — the IoU-only
baseline tracker that ships with Ultralytics.

## ByteTrack vs BoTSORT — Key Differences

| Feature | BoTSORT (04) | ByteTrack (04b) |
|---|---|---|
| Motion model | Kalman + IoU | IoU only |
| Global motion compensation | `gmc_method: sparseOptFlow` | None |
| Re-ID / appearance | Optional (boxmot) | **Not supported** |
| Speed | Slower (ReID pass) | **Faster** |

> **Result:** ByteTrack produces more fragments than BoTSORT+ReID but runs significantly
> faster. It remains a viable option when speed matters more than count accuracy.

## 0 — Configuration

In [ ]:
from pathlib import Path
import json, cv2, yaml, itertools, subprocess, sys, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ultralytics import YOLO

REPO_ROOT    = Path('/home1/hendersonj6179@cgu.edu')
RUNS_DIR     = (REPO_ROOT / 'runs').resolve()
TRACKER_RUNS = (REPO_ROOT / 'runs/trackers/bytetrack_tuning').resolve()
TRACKER_RUNS.mkdir(parents=True, exist_ok=True)

# ── Detector weights ────────────────────────────────────────────────────────
WEIGHTS    = RUNS_DIR / 'yolo26' / 'final' / 'weights' / 'best.pt'
# WEIGHTS  = RUNS_DIR / 'rtdetr' / 'final' / 'weights' / 'best.pt'
CHUNKS_DIR = REPO_ROOT / 'data/processed/labelstudiochunks'

# Detection confidence for all tracker runs in this notebook
CONF = 0.25

with open(REPO_ROOT / 'data/splits/splits.json') as f:
    splits = json.load(f)
CLASSES = splits['classes']
N_CLS   = len(CLASSES)

test_videos = sorted(CHUNKS_DIR.glob('*.mp4'))
TEST_VIDEO  = test_videos[0] if test_videos else None
print(f'Test video : {TEST_VIDEO}')
print(f'Classes    : {CLASSES}')
print(f'Weights    : {WEIGHTS}')


## 1 — Track Fragmentation Diagnostics

Run the baseline ByteTrack and plot a **track timeline** — one horizontal bar per
Track ID, coloured by class.

- **Gaps** = frames where the tracker lost the object
- **Multiple short same-colour bars** = fragmentation (same icon, multiple IDs)

In [ ]:
from helpers import plot_track_timeline

## 1b — Ground-Truth Timeline & Fragment Counts

Loads the MOT-format `gt/gt.txt` and plots the same track-timeline for reference.
Comparing GT vs baseline shows the upper bound achievable by a perfect tracker.

In [ ]:
from helpers import load_gt_as_df, count_fragments

## 2 — ByteTrack Parameter Sweep

Sweeps the knobs available in ByteTrack. Because there is **no ReID or GMC**,
`track_buffer` and `match_thresh` are the primary levers.

| Parameter | Effect |
|---|---|
| `track_buffer` | Frames a lost track stays alive. **Raise** to bridge re-entry gaps. |
| `match_thresh` | IoU threshold for association. **Lower** (0.5–0.6) to allow positional jumps. |
| `new_track_thresh` | Min score to spawn a new track. **Raise** to suppress spurious IDs. |

In [ ]:
def write_bytetrack_cfg(path, track_buffer, match_thresh,
                        new_track_thresh=0.30,
                        track_high_thresh=0.25, track_low_thresh=0.10,
                        fuse_score=True):
    """Write a ByteTrack YAML config.

    ByteTrack supports exactly these keys:
      tracker_type, track_high_thresh, track_low_thresh,
      new_track_thresh, track_buffer, match_thresh, fuse_score
    No ReID, no GMC, no appearance_thresh.
    """
    cfg = {
        'tracker_type':      'bytetrack',
        'track_high_thresh': track_high_thresh,
        'track_low_thresh':  track_low_thresh,
        'new_track_thresh':  new_track_thresh,
        'track_buffer':      track_buffer,
        'match_thresh':      match_thresh,
        'fuse_score':        fuse_score,
    }
    Path(path).write_text(yaml.dump(cfg))


# Sweep grid — mirrors Section 2 of notebook 07 where applicable
# new_track_thresh also swept here (no ReID section to compensate)
SWEEP_BT = {
    'track_buffer':      [10, 20, 45],
    'match_thresh':      [0.50, 0.60, 0.80, 0.99],
    'new_track_thresh':  [0.25, 0.40, 0.55],
}

bt_results = []
if TEST_VIDEO:
    for tb, mt, ntt in itertools.product(
            SWEEP_BT['track_buffer'],
            SWEEP_BT['match_thresh'],
            SWEEP_BT['new_track_thresh']):
        cfg_path = TRACKER_RUNS / f'bt_tb{tb}_mt{int(mt*100)}_ntt{int(ntt*100)}.yaml'
        write_bytetrack_cfg(cfg_path, tb, mt, new_track_thresh=ntt)
        df_sw = run_tracker(TEST_VIDEO, WEIGHTS, str(cfg_path), conf=CONF)
        frags = count_fragments(df_sw)
        total = sum(frags.values())
        bt_results.append({'track_buffer': tb, 'match_thresh': mt,
                           'new_track_thresh': ntt, 'total_fragments': total, **frags})
        print(f'  tb={tb:2d}  mt={mt:.2f}  ntt={ntt:.2f}  -> fragments={total}')
    print('\nTop 5 (ByteTrack sweep):')
    print(pd.DataFrame(bt_results).sort_values('total_fragments').head(5).to_string(index=False))
else:
    print('Set TEST_VIDEO to run.')


## 3 — Comparison: GT vs Baseline vs Best ByteTrack

Fragment count bar chart comparing ground truth, baseline ByteTrack (default config),
and best ByteTrack (best sweep result).

In [ ]:
if bt_results:
    best_bt = pd.DataFrame(bt_results).sort_values('total_fragments').iloc[0]
    print('=== Best ByteTrack config ===')
    print(f'  tb={int(best_bt.track_buffer)}  mt={best_bt.match_thresh:.2f}'
          f'  ntt={best_bt.new_track_thresh:.2f}  fragments={int(best_bt.total_fragments)}')

    rows_to_plot = []
    if 'df_gt' in dir() and not df_gt.empty:
        rows_to_plot.append(('Ground Truth', count_fragments(df_gt), 'black'))
    rows_to_plot.append(('Baseline ByteTrack', count_fragments(df_base), 'steelblue'))
    rows_to_plot.append(('Best ByteTrack', {c: int(best_bt.get(c, 0)) for c in CLASSES}, 'darkorange'))

    x = np.arange(N_CLS); w = 0.8 / len(rows_to_plot)
    fig, ax = plt.subplots(figsize=(15, 5))
    for k, (label, frags, color) in enumerate(rows_to_plot):
        offset = (k - len(rows_to_plot) / 2 + 0.5) * w
        ax.bar(x + offset, [frags.get(c, 0) for c in CLASSES],
               w, label=label, color=color, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30, ha='right')
    ax.set_ylabel('Track fragments (lower = better)')
    ax.set_title('Fragment Count: GT vs ByteTrack Configurations', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
    out = TRACKER_RUNS / 'bytetrack_comparison.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {out}')
else:
    print('Run Section 2 first.')


## 4 — Fragment Merge Post-Processor

Because ByteTrack has no ReID, post-processing is especially important.
`merge_fragments()` stitches same-class fragments that are temporally close
and spatially nearby, bridging gaps the tracker could not.

**Only the single closest candidate is merged** per ended track.

In [ ]:
from helpers import merge_fragments

## 5 — Multi-Video Comparison: GT vs Baseline vs Best ByteTrack (No Merge)

Runs both tracker variants across `N_SAMPLE` video chunks and reports fragment
counts + elapsed time. No post-processing merge applied.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
N_SAMPLE = 5
CONF_5   = 0.25
SEED     = 42

random.seed(SEED)
all_videos    = sorted(CHUNKS_DIR.glob('*.mp4'))
sample_videos = random.sample(all_videos, min(N_SAMPLE, len(all_videos)))
print(f'Sampled {len(sample_videos)} videos (conf={CONF_5}):')
for v in sample_videos: print(f'  {v.name}')

# Best config YAML (falls back to defaults if sweep not run)
def _best_row(results, defaults):
    """Pick the sweep config with the fewest total fragments (sum across all classes)."""
    if results:
        return pd.DataFrame(results).sort_values('total_fragments').iloc[0].to_dict()
    return defaults

bt_best = _best_row(bt_results if 'bt_results' in dir() else [],
                    {'track_buffer': 30, 'match_thresh': 0.60, 'new_track_thresh': 0.30})

_baseline_cfg = TRACKER_RUNS / 'baseline.yaml'
write_bytetrack_cfg(_baseline_cfg, track_buffer=30, match_thresh=0.8)

_best_cfg = TRACKER_RUNS / 'best.yaml'
write_bytetrack_cfg(_best_cfg,
                    track_buffer=int(bt_best['track_buffer']),
                    match_thresh=float(bt_best['match_thresh']),
                    new_track_thresh=float(bt_best.get('new_track_thresh', 0.30)))

per_video_tables_5 = []

for vid in sample_videos:
    vname = vid.stem
    print(f'\n{"="*55}\nVideo: {vname}\n{"="*55}')

    df_gt_v, _ = load_gt_as_df(vid, MOT_SEQ_DIR)
    gt_frags = count_fragments(df_gt_v) if not df_gt_v.empty else {c: 0 for c in CLASSES}

    t0 = time.time(); df_bl = run_tracker(vid, WEIGHTS, str(_baseline_cfg), conf=CONF_5); t_bl = time.time() - t0
    bl_frags = count_fragments(df_bl)

    t0 = time.time(); df_bst = run_tracker(vid, WEIGHTS, str(_best_cfg), conf=CONF_5); t_bst = time.time() - t0
    bst_frags = count_fragments(df_bst)

    tbl = pd.DataFrame({
        'Ground Truth':       [gt_frags.get(c, 0)  for c in CLASSES],
        'Baseline ByteTrack': [bl_frags.get(c, 0)  for c in CLASSES],
        'Best ByteTrack':     [bst_frags.get(c, 0) for c in CLASSES],
    }, index=CLASSES)
    tbl.loc['TOTAL']    = tbl.sum()
    tbl.loc['Time (s)'] = [None, round(t_bl, 1), round(t_bst, 1)]
    per_video_tables_5.append((vname, tbl))

    print(f'\nFragment counts — {vname}:')
    display(tbl.style
               .highlight_min(axis=1, color='#c6efce',
                              subset=pd.IndexSlice[CLASSES, ['Baseline ByteTrack', 'Best ByteTrack']])
               .format(lambda x: f'{x:.1f}' if isinstance(x, float) and not pd.isna(x)
                       else (str(int(x)) if x is not None and not (isinstance(x, float) and pd.isna(x)) else '—')))

print(f'\n{"="*55}\nAGGREGATE (no post-merge)\n{"="*55}')
agg5 = pd.DataFrame({
    col: [sum(tbl.loc[c, col] or 0 for _, tbl in per_video_tables_5
              if c in tbl.index and tbl.loc[c, col] is not None)
          for c in CLASSES]
    for col in ['Ground Truth', 'Baseline ByteTrack', 'Best ByteTrack']
}, index=CLASSES)
agg5.loc['TOTAL'] = agg5.sum()
display(agg5.style.highlight_min(axis=1, color='#c6efce',
        subset=['Baseline ByteTrack', 'Best ByteTrack']))

agg5.to_csv(TRACKER_RUNS / 'multi_video_aggregate_raw.csv')
print(f'Saved: {TRACKER_RUNS / "multi_video_aggregate_raw.csv"}')


## 5b — Multi-Video Comparison: GT vs Trackers (with Post-Processing Merge)

Same as Section 5 but applies `merge_fragments()` before counting.

**Configuration knobs:**
- `MAX_GAP` — max frame gap to bridge
- `MAX_DIST_PCT` — max centroid distance as % of frame diagonal

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
N_SAMPLE_B   = 5
CONF_5B      = 0.25
MAX_GAP      = 45
MAX_DIST_PCT = 35.0
SEED_B       = 42

random.seed(SEED_B)
sample_videos_b = random.sample(all_videos, min(N_SAMPLE_B, len(all_videos)))
print(f'Sampled {len(sample_videos_b)} videos (conf={CONF_5B}, max_gap={MAX_GAP}, max_dist={MAX_DIST_PCT}%):')
for v in sample_videos_b: print(f'  {v.name}')

per_video_tables_5b = []

for vid in sample_videos_b:
    vname = vid.stem
    print(f'\n{"="*55}\nVideo: {vname}\n{"="*55}')

    df_gt_v, _ = load_gt_as_df(vid, MOT_SEQ_DIR)
    gt_frags = count_fragments(df_gt_v) if not df_gt_v.empty else {c: 0 for c in CLASSES}

    t0 = time.time()
    df_bl_b = run_tracker(vid, WEIGHTS, str(_baseline_cfg), conf=CONF_5B)
    df_bl_m = merge_fragments(df_bl_b, max_gap=MAX_GAP, max_dist_pct=MAX_DIST_PCT)
    t_bl_b = time.time() - t0
    bl_frags_b = count_fragments(df_bl_m)

    t0 = time.time()
    df_bst_b = run_tracker(vid, WEIGHTS, str(_best_cfg), conf=CONF_5B)
    df_bst_m = merge_fragments(df_bst_b, max_gap=MAX_GAP, max_dist_pct=MAX_DIST_PCT)
    t_bst_b = time.time() - t0
    bst_frags_b = count_fragments(df_bst_m)

    tbl = pd.DataFrame({
        'Ground Truth':           [gt_frags.get(c, 0)   for c in CLASSES],
        'Baseline+Merge':         [bl_frags_b.get(c, 0)  for c in CLASSES],
        'Best ByteTrack+Merge':   [bst_frags_b.get(c, 0) for c in CLASSES],
    }, index=CLASSES)
    tbl.loc['TOTAL']    = tbl.sum()
    tbl.loc['Time (s)'] = [None, round(t_bl_b, 1), round(t_bst_b, 1)]
    per_video_tables_5b.append((vname, tbl))

    print(f'\nFragment counts (post-merge) — {vname}:')
    display(tbl.style
               .highlight_min(axis=1, color='#c6efce',
                              subset=pd.IndexSlice[CLASSES, ['Baseline+Merge', 'Best ByteTrack+Merge']])
               .format(lambda x: f'{x:.1f}' if isinstance(x, float) and not pd.isna(x)
                       else (str(int(x)) if x is not None and not (isinstance(x, float) and pd.isna(x)) else '—')))

print(f'\n{"="*55}\nAGGREGATE (post-merge)\n{"="*55}')
agg5b = pd.DataFrame({
    col: [sum(tbl.loc[c, col] or 0 for _, tbl in per_video_tables_5b
              if c in tbl.index and tbl.loc[c, col] is not None)
          for c in CLASSES]
    for col in ['Ground Truth', 'Baseline+Merge', 'Best ByteTrack+Merge']
}, index=CLASSES)
agg5b.loc['TOTAL'] = agg5b.sum()
display(agg5b.style.highlight_min(axis=1, color='#c6efce',
        subset=['Baseline+Merge', 'Best ByteTrack+Merge']))

agg5b.to_csv(TRACKER_RUNS / 'multi_video_aggregate_merged.csv')
print(f'Saved: {TRACKER_RUNS / "multi_video_aggregate_merged.csv"}')


## 6 — Save Best Architecture Config

Writes two files:
- `configs/kartector_bytetrack_best.yaml` — tracker knobs for Ultralytics
- `configs/kartector_bytetrack_arch.json` — inference + post-process settings

This JSON is read by notebooks 07 and 08 when `ACTIVE_TRACKER = 'bytetrack'`.

In [ ]:
if bt_results:
    best = pd.DataFrame(bt_results).sort_values('total_fragments').iloc[0]

    # ── Tracker YAML (Ultralytics format) ─────────────────────────────────────
    best_cfg_dict = {
        'tracker_type':      'bytetrack',
        'track_high_thresh': 0.25,
        'track_low_thresh':  0.10,
        'new_track_thresh':  float(best.get('new_track_thresh', 0.30)),
        'track_buffer':      int(best['track_buffer']),
        'match_thresh':      float(best['match_thresh']),
        'fuse_score':        True,
    }
    out_cfg = REPO_ROOT / 'configs' / 'kartector_bytetrack_best.yaml'
    out_cfg.write_text(yaml.dump(best_cfg_dict))
    print(f'Tracker YAML written to: {out_cfg}')
    print(yaml.dump(best_cfg_dict))

    # ── Architecture JSON (inference + post-process settings) ─────────────────
    post_process_on = 'per_video_tables_5b' in dir() and bool(per_video_tables_5b)
    arch_max_gap      = MAX_GAP      if 'MAX_GAP'      in dir() else 45
    arch_max_dist_pct = MAX_DIST_PCT if 'MAX_DIST_PCT' in dir() else 35.0

    arch = {
        'tracker':      'bytetrack',
        'tracker_yaml': str(out_cfg.relative_to(REPO_ROOT)),
        'conf':         float(CONF),
        'post_process': post_process_on,
        'max_gap':      int(arch_max_gap),
        'max_dist_pct': float(arch_max_dist_pct),
    }
    import json as _json
    arch_out = REPO_ROOT / 'configs' / 'kartector_bytetrack_arch.json'
    arch_out.write_text(_json.dumps(arch, indent=2))
    print(f'Architecture config written to: {arch_out}')
    print(_json.dumps(arch, indent=2))
else:
    print('Run Section 2 first.')
